In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import yaml
from PIL import Image
from datetime import datetime

c1_image_file = r"C:\Users\cvolp\shorescan_processing\DATA\CACO04_c1.jpg"
c2_image_file = r"C:\Users\cvolp\shorescan_processing\DATA\CACO04_c2.jpg"

c1_timestack_file = "C:/cvolpano/shorescan_processing/CACO03_c1_20241023_timestack.tiff"
c2_timestack_file = "C:/cvolpano/shorescan_processing/CACO03_c2_20241023_timestack.tiff"

transect_file_c1 = r"E:\runup-Chelsea\DATA\YAML\CACO04_c1_timestack_20250123.yaml"
transect_file_c2 = r"E:\runup-Chelsea\DATA\YAML\CACO04_c2_timestack_20250123.yaml"

pix_file_c1 = r"E:\runup-Chelsea\DATA\CACO03_c1_timestack_20250123.pix"
pix_file_c2 = r"E:\runup-Chelsea\DATA\CACO03_c2_timestack_20250123.pix"
output_dir = "C:/cvolpano/shorescan_processing/results/output_plots/"

# SITE (4 letters + 2 digits) _ camera (c1/c2) _ date (8 digits)
FNAME_PATTERN = re.compile(
    r"(?P<site>[A-Za-z]{4}\d{2})_(?P<cam>[cC][12])_(?P<date>\d{8})"
)

def parse_shorescan_name(filepath):
    """
    Extract site, camera, and date from a ShoreScan-style filepath.
    filepath : str or pathlib.Path
        e.g. "C:/.../CACO03_c1_20241023_GCP_image_raw.jpg"
    Returns
    dict
        {'site': 'CACO03', 'camera': 'c1', 'date': datetime(2024, 10, 23)}
    Raises ValueError
        If the filename doesn't match, or the date isn't a real date.
    """
    name = Path(filepath).name
    match = FNAME_PATTERN.search(name)
    if match is None:
        raise ValueError(f"Filename does not match SITE_cX_YYYYMMDD: {name}")

    date = datetime.strptime(match["date"], "%Y%m%d")
    return {
        "site": match["site"].upper(),
        "camera": match["cam"].lower(),
        "date": date,
    }

def plot_transects(image_file, transect_file, colormap='magma'):
    with open(transect_file, "r") as f:
        data = yaml.safe_load(f)\

    dfs = {}
    for transect, values in data.items():
        cols = {}
        for key, value in values.items():
            if value is None or value == "":
                continue
            # Convert comma-separated string to numeric array
            cols[key] = np.array(
                [float(x) for x in str(value).split(",") if x.strip()]
            )
        dfs[transect] = pd.DataFrame(cols)
    
    img = np.array(Image.open(image_file))
    n_colors = len(transect)
    cmap = mpl.colormaps[colormap].resampled(n_colors)

    transect_fparts = parse_shorescan_name(transect_file)
    camera_fparts = parse_shorescan_name(image_file)

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(img)
    count =0
    for transect, values in data.items():
        # Handle empty fields (e.g., V:)
        U = np.array([float(x) for x in values['U'].split(',') if x.strip()])
        V = np.array([float(x) for x in values['V'].split(',') if x.strip()])
        if len(U) == 0 or len(V) == 0:
            print(f"Skipping {transect}: empty U or V")
            continue
        ax.plot(U, V, '-', lw=2,c=cmap(count))

        label = transect.split('_')[-1]  
        ax.text(
            U[(len(U) // 2)],
            V[(len(V) // 2)+10],
            label,
            fontsize=10,
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none')
        )
        count=count+1

    ax.set_aspect('equal')
    ax.set_title(f"{camera_fparts['site'].upper()} ({camera_fparts['camera'].upper()}) \n {transect_fparts['date'].strftime('%Y-%m-%d')}", fontsize=14)
    ax.set_xlabel("u (pixels)")
    ax.set_ylabel("v (pixels)")
    plt.tight_layout()
    plt.savefig(f'{output_dir}Transects_{transect_fparts["date"].strftime("%Y%m%d")}_{transect_fparts["camera"]}_{camera_fparts["date"].strftime("%Y%m%d")}.png')
    plt.show()

# plot_transects(c1_image_file, transect_file_c1, colormap='magma')
# plot_transects(c2_image_file, transect_file_c2, colormap='magma')